In [4]:
from sklearn.model_selection import KFold
import numpy as np
import pandas as pd

from factor_analyzer import FactorAnalyzer
from sklearn.mixture import GaussianMixture

In [ ]:
# 0. CARGAR DATASET
# =======================================
# df = pd.read_csv("../data_collection_peru_csv/data_merge/COLLECT_COMP_PERU.csv")
df = pd.read_csv("../data_ready/COLLECT_COMP_dataset_treated.csv")
df_afe = pd.read_csv("../Aprendisaje/df_afe.csv")
df_acl_scaled = pd.read_csv("../Aprendisaje/df_acl_scaled.csv")
# df_with_factors = pd.read_csv('../Aprendisaje/df_with_factors.csv')
df_with_classes = pd.read_csv('../Aprendisaje/df_with_classes.csv')



In [ ]:
n_factors = 6
n_classes_final = 9

In [19]:
print("="*80)
print("INTEGRACIÓN DE RESULTADOS: VARIABLES LATENTES Y RENDIMIENTO ACADÉMICO")
print("="*80)

# 1. RESUMEN DE FACTORES ENCONTRADOS
print("\n1. FACTORES LATENTES IDENTIFICADOS (AFE):")
print("-"*40)
for i in range(n_factors):
    variance_explained = results['varimax']['variance'][1][i] * 100
    print(f"\nFactor {i+1} ({variance_explained:.1f}% varianza):")
    print("  Interpretación: [Basado en ítems con alta carga]")
    print("  Ítems principales: [Listar top 3 ítems]")

# 2. PERFILES DE ESTUDIANTES
print("\n2. PERFILES DE ESTUDIANTES (ACL):")
print("-"*40)
for clase in range(n_classes_final):
    n_students = (df_with_classes['latent_class'] == clase).sum()
    pct = (n_students / len(df_with_classes)) * 100
    
    print(f"\nPerfil {clase+1} ({pct:.1f}% de estudiantes):")
    
    # Características principales
    clase_data = df_with_classes[df_with_classes['latent_class'] == clase]
    
    # Rendimiento
    if 'PV1MATH' in clase_data.columns:
        math_mean = clase_data['PV1MATH'].mean()
        math_std = clase_data['PV1MATH'].std()
        print(f"  Rendimiento: {math_mean:.0f} ± {math_std:.0f}")
    
    # Factores
    for factor in [col for col in clase_data.columns if 'Factor_' in col]:
        factor_mean = clase_data[factor].mean()
        if factor_mean > 0.3:
            print(f"  {factor}: Alto")
        elif factor_mean < -0.3:
            print(f"  {factor}: Bajo")
    
    print(f"  Caracterización: [Descripción del perfil]")

# 3. PREDICTORES CLAVE DEL RENDIMIENTO
print("\n3. PREDICTORES CLAVE DEL RENDIMIENTO:")
print("-"*40)

# Modelo de regresión para integrar resultados
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Variables predictoras
predictors = [col for col in df_with_classes.columns if 'Factor_' in col]
predictors.append('theta_irt')  # Añadir habilidad TRI

# Crear dummies para clases latentes
class_dummies = pd.get_dummies(df_with_classes['latent_class'], prefix='clase')
X = pd.concat([df_with_classes[predictors], class_dummies], axis=1)
X = X.dropna()

# Variable objetivo
y = df_with_classes.loc[X.index, 'PV1MATH']

# Ajustar modelo
model = LinearRegression()
model.fit(X, y)

# Importancia de variables
feature_importance = pd.DataFrame({
    'Variable': X.columns,
    'Coeficiente': model.coef_,
    'Abs_Coef': np.abs(model.coef_)
}).sort_values('Abs_Coef', ascending=False)

print("\nVariables más importantes para el rendimiento:")
for _, row in feature_importance.head(10).iterrows():
    print(f"  {row['Variable']}: β = {row['Coeficiente']:.2f}")

r2 = r2_score(y, model.predict(X))
print(f"\nVarianza explicada (R²): {r2:.3f}")

# 4. RECOMENDACIONES PEDAGÓGICAS
print("\n4. RECOMENDACIONES BASADAS EN EL ANÁLISIS:")
print("-"*40)

recommendations = {
    "Alto rendimiento": [],
    "Rendimiento medio": [],
    "Bajo rendimiento": []
}

for clase in range(n_classes_final):
    clase_data = df_with_classes[df_with_classes['latent_class'] == clase]
    mean_performance = clase_data['PV1MATH'].mean() if 'PV1MATH' in clase_data.columns else 0
    
    if mean_performance > df_with_classes['PV1MATH'].quantile(0.66):
        category = "Alto rendimiento"
    elif mean_performance > df_with_classes['PV1MATH'].quantile(0.33):
        category = "Rendimiento medio"
    else:
        category = "Bajo rendimiento"
    
    # Generar recomendaciones basadas en características
    recommendations[category].append(f"Clase {clase+1}")

for category, classes in recommendations.items():
    print(f"\n{category} ({', '.join(classes)}):")
    print("  • [Recomendaciones específicas basadas en los factores]")
    print("  • [Estrategias de intervención sugeridas]")

# 5. VISUALIZACIÓN INTEGRADORA FINAL
fig, axes = plt.subplots(2, 2, figsize=(15, 12))

# Panel 1: Factores por clase
ax1 = axes[0, 0]
factor_means = df_with_classes.groupby('latent_class')[[col for col in df_with_classes.columns if 'Factor_' in col]].mean()
factor_means.T.plot(kind='bar', ax=ax1)
ax1.set_title('Factores Promedio por Clase Latente')
ax1.set_xlabel('Factor')
ax1.set_ylabel('Puntuación Media')
ax1.legend(title='Clase')

# Panel 2: Rendimiento por clase
ax2 = axes[0, 1]
df_with_classes.boxplot(column='PV1MATH', by='latent_class', ax=ax2)
ax2.set_title('Rendimiento Académico por Clase Latente')
ax2.set_xlabel('Clase Latente')
ax2.set_ylabel('Puntaje PISA')

# Panel 3: Habilidad TRI vs Rendimiento
ax3 = axes[1, 0]
ax3.scatter(df_with_classes['theta_irt'], df_with_classes['PV1MATH'], 
           c=df_with_classes['latent_class'], alpha=0.6, cmap='viridis')
ax3.set_xlabel('Habilidad TRI (θ)')
ax3.set_ylabel('Rendimiento Matemático')
ax3.set_title('Relación Habilidad-Rendimiento por Clase')

# Panel 4: Función de información por clase
ax4 = axes[1, 1]
for clase in range(n_classes_final):
    clase_mask = df_with_classes['latent_class'] == clase
    if 'theta_irt' in df_with_classes.columns:
        theta_clase = df_with_classes.loc[clase_mask, 'theta_irt'].dropna()
        if len(theta_clase) > 0:
            ax4.hist(theta_clase, alpha=0.5, label=f'Clase {clase}', bins=20)
ax4.set_xlabel('Habilidad (θ)')
ax4.set_ylabel('Frecuencia')
ax4.set_title('Distribución de Habilidad por Clase')
ax4.legend()

plt.suptitle('Análisis Integrado: Variables Latentes y Rendimiento Académico', fontsize=16)
plt.tight_layout()
plt.show()

# GUARDAR DATASET FINAL
output_file = 'puno_students_latent_analysis.csv'
df_with_classes.to_csv(output_file, index=False)
print(f"\n✅ Dataset final guardado como: {output_file}")
print(f"   Dimensiones: {df_with_classes.shape}")
print(f"   Nuevas variables añadidas:")
print(f"   - {n_factors} puntuaciones factoriales")
print(f"   - 1 variable de clase latente")
print(f"   - 1 puntuación de habilidad TRI")

INTEGRACIÓN DE RESULTADOS: VARIABLES LATENTES Y RENDIMIENTO ACADÉMICO

1. FACTORES LATENTES IDENTIFICADOS (AFE):
----------------------------------------


NameError: name 'results' is not defined